In [4]:
# Install all required libraries
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install langchain-text-splitters
!pip install sentence-transformers
!pip install chromadb
!pip install faiss-cpu
!pip install transformers
!pip install accelerate
!pip install bitsandbytes
!pip install datasets
!pip install huggingface_hub

In [5]:
import os
import warnings
warnings.filterwarnings('ignore')

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

In [6]:
# Generate a comprehensive mental health knowledge base dataset
mental_health_data = [
    # Anxiety
    {
        "topic": "Anxiety Disorders",
        "content": """Anxiety disorders are characterized by excessive fear, worry, and related behavioral disturbances.
        Symptoms include restlessness, fatigue, difficulty concentrating, irritability, muscle tension, and sleep problems.
        Common types include Generalized Anxiety Disorder (GAD), Panic Disorder, Social Anxiety Disorder, and Specific Phobias.
        Treatment options include Cognitive Behavioral Therapy (CBT), exposure therapy, and medications like SSRIs and benzodiazepines.
        Self-help strategies include deep breathing exercises, progressive muscle relaxation, mindfulness meditation, and regular physical exercise."""
    },
    {
        "topic": "Panic Attacks",
        "content": """Panic attacks are sudden episodes of intense fear that trigger severe physical reactions.
        Symptoms include rapid heartbeat, sweating, trembling, shortness of breath, chest pain, nausea, dizziness, and fear of losing control.
        Panic attacks typically peak within 10 minutes and rarely last more than 30 minutes.
        Grounding techniques like the 5-4-3-2-1 method can help: identify 5 things you see, 4 you hear, 3 you touch, 2 you smell, 1 you taste.
        Long-term management includes CBT, medication, and lifestyle changes like reducing caffeine and alcohol."""
    },
    # Depression
    {
        "topic": "Depression",
        "content": """Depression (Major Depressive Disorder) is a mood disorder causing persistent feelings of sadness and loss of interest.
        Symptoms include depressed mood, loss of interest in activities, changes in appetite and weight, sleep disturbances, fatigue,
        feelings of worthlessness or guilt, difficulty thinking or concentrating, and thoughts of death or suicide.
        For diagnosis, symptoms must persist for at least two weeks and represent a change from previous functioning.
        Treatment includes psychotherapy (especially CBT and interpersonal therapy), antidepressant medications (SSRIs, SNRIs, TCAs),
        and in severe cases, electroconvulsive therapy (ECT). Lifestyle factors like exercise, sleep hygiene, and social support are crucial."""
    },
    {
        "topic": "Seasonal Affective Disorder",
        "content": """Seasonal Affective Disorder (SAD) is a type of depression related to changes in seasons, typically starting in fall and continuing through winter.
        Symptoms include low energy, oversleeping, overeating, weight gain, craving carbohydrates, and social withdrawal.
        The condition is linked to reduced sunlight exposure affecting serotonin levels and melatonin balance.
        Treatment options include light therapy (phototherapy) using a 10,000 lux light box for 20-30 minutes daily,
        psychotherapy, antidepressant medications, and vitamin D supplementation. Dawn simulators can also help regulate sleep-wake cycles."""
    },
    # PTSD
    {
        "topic": "Post-Traumatic Stress Disorder",
        "content": """PTSD develops after experiencing or witnessing a traumatic event such as combat, assault, accidents, or natural disasters.
        Symptoms are grouped into four categories: intrusive memories (flashbacks, nightmares), avoidance behaviors,
        negative changes in thinking and mood, and changes in physical and emotional reactions (hyperarousal).
        Symptoms must last more than one month and cause significant distress for PTSD diagnosis.
        Evidence-based treatments include Prolonged Exposure therapy, Cognitive Processing Therapy (CPT), and Eye Movement Desensitization
        and Reprocessing (EMDR). Medications like SSRIs (sertraline, paroxetine) are FDA-approved for PTSD treatment."""
    },
    # Stress Management
    {
        "topic": "Stress Management Techniques",
        "content": """Chronic stress can lead to various health problems including anxiety, depression, heart disease, and weakened immune system.
        Effective stress management techniques include: mindfulness meditation (focusing on present moment awareness),
        progressive muscle relaxation (tensing and releasing muscle groups), deep breathing exercises (diaphragmatic breathing),
        regular physical exercise (at least 150 minutes of moderate activity weekly), adequate sleep (7-9 hours for adults),
        time management and prioritization, social support and connection, limiting caffeine and alcohol,
        and engaging in hobbies and enjoyable activities. The stress response can be managed through the relaxation response technique."""
    },
    # Sleep Disorders
    {
        "topic": "Sleep and Mental Health",
        "content": """Sleep and mental health are closely connected. Poor sleep can contribute to mental health problems, and mental health conditions often disrupt sleep.
        Insomnia affects 30-40% of adults and is common in depression, anxiety, and PTSD.
        Good sleep hygiene practices include: maintaining consistent sleep-wake times, creating a dark and cool sleep environment,
        avoiding screens 1-2 hours before bed, limiting caffeine after noon, avoiding alcohol close to bedtime,
        regular exercise (but not close to bedtime), and using the bed only for sleep and intimacy.
        Cognitive Behavioral Therapy for Insomnia (CBT-I) is the first-line treatment and includes sleep restriction, stimulus control, and cognitive restructuring."""
    },
    # Mindfulness
    {
        "topic": "Mindfulness and Meditation",
        "content": """Mindfulness is the practice of purposely focusing attention on the present moment without judgment.
        Research shows mindfulness reduces symptoms of anxiety, depression, and chronic pain.
        Types of mindfulness practices include: focused attention meditation (concentrating on breath or object),
        body scan meditation (progressively focusing on different body parts), loving-kindness meditation (cultivating compassion),
        and mindful movement (yoga, tai chi, walking meditation).
        Start with just 5-10 minutes daily and gradually increase. Apps like Headspace, Calm, and Insight Timer can guide beginners.
        Mindfulness-Based Stress Reduction (MBSR) is an 8-week program with strong evidence for various conditions."""
    },
    # Therapy Types
    {
        "topic": "Types of Psychotherapy",
        "content": """Psychotherapy involves talking with a mental health professional to treat mental health conditions.
        Cognitive Behavioral Therapy (CBT) focuses on identifying and changing negative thought patterns and behaviors. It's effective for depression, anxiety, and many other conditions.
        Dialectical Behavior Therapy (DBT) combines CBT with mindfulness and is especially effective for borderline personality disorder and self-harm behaviors.
        Psychodynamic therapy explores unconscious patterns and past experiences affecting current behavior.
        Interpersonal therapy focuses on improving communication and relationship patterns.
        Acceptance and Commitment Therapy (ACT) emphasizes accepting thoughts and feelings while committing to value-based actions.
        The therapeutic relationship is a key factor in treatment success regardless of the specific approach used."""
    },
    # Crisis Resources
    {
        "topic": "Mental Health Crisis Resources",
        "content": """A mental health crisis requires immediate attention. Warning signs include thoughts of suicide or self-harm,
        severe anxiety or panic attacks, psychotic symptoms, or inability to care for oneself.
        In the US, the 988 Suicide and Crisis Lifeline provides 24/7 support (call or text 988).
        Crisis Text Line: Text HOME to 741741. Veterans Crisis Line: 1-800-273-8255, Press 1.
        If someone is in immediate danger, call 911 or go to the nearest emergency room.
        Safety planning involves identifying warning signs, coping strategies, people to contact, and removing access to lethal means.
        After a crisis, follow-up care and developing a safety plan are essential for prevention."""
    },
    # Social Anxiety
    {
        "topic": "Social Anxiety Disorder",
        "content": """Social Anxiety Disorder involves intense fear of social situations where one might be judged, embarrassed, or humiliated.
        Physical symptoms include blushing, sweating, trembling, rapid heartbeat, nausea, and difficulty speaking.
        People with social anxiety often avoid social situations or endure them with intense distress.
        Common fears include meeting new people, being the center of attention, being watched while doing something, and making small talk.
        Treatment includes CBT with exposure therapy, social skills training, and medications (SSRIs, beta-blockers for performance anxiety).
        Gradual exposure to feared situations, starting with less threatening scenarios, helps build confidence over time."""
    },
    # Self-Care
    {
        "topic": "Self-Care for Mental Health",
        "content": """Self-care involves activities that promote physical, emotional, and mental well-being.
        Physical self-care includes regular exercise, balanced nutrition, adequate sleep, and attending medical appointments.
        Emotional self-care involves acknowledging feelings, practicing self-compassion, setting boundaries, and seeking support when needed.
        Social self-care means maintaining healthy relationships, spending time with supportive people, and asking for help.
        Mental self-care includes engaging in stimulating activities, learning new skills, practicing mindfulness, and limiting news consumption.
        Spiritual self-care might involve meditation, spending time in nature, practicing gratitude, or engaging with a faith community.
        Self-care is not selfish; it's necessary for maintaining mental health and being able to care for others."""
    },
]

# Create DataFrame
df = pd.DataFrame(mental_health_data)
print(f"Created Mental Health Dataset with {len(df)} documents")
display(df)

Created Mental Health Dataset with 12 documents


,topic,content
0,Anxiety Disorders,Anxiety disorders are characterized by excessi...
1,Panic Attacks,Panic attacks are sudden episodes of intense f...
2,Depression,Depression (Major Depressive Disorder) is a mo...
3,Seasonal Affective Disorder,Seasonal Affective Disorder (SAD) is a type of...
4,Post-Traumatic Stress Disorder,PTSD develops after experiencing or witnessing...
5,Stress Management Techniques,Chronic stress can lead to various health prob...
6,Sleep and Mental Health,Sleep and mental health are closely connected....
7,Mindfulness and Meditation,Mindfulness is the practice of purposely focus...
8,Types of Psychotherapy,Psychotherapy involves talking with a mental h...
9,Mental Health Crisis Resources,A mental health crisis requires immediate atte...


In [7]:
# Convert data to LangChain Documents
documents = []
for idx, row in df.iterrows():
    doc = Document(
        page_content=f"Topic: {row['topic']}\n\n{row['content']}",
        metadata={"topic": row["topic"], "doc_id": idx}
    )
    documents.append(doc)

print(f"Total documents: {len(documents)}")

# Text Chunking with different chunk sizes
def chunk_documents(docs, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_documents(docs)

# Create chunks with different sizes for comparison
chunks_small = chunk_documents(documents, chunk_size=256, chunk_overlap=50)
chunks_medium = chunk_documents(documents, chunk_size=512, chunk_overlap=100)
chunks_large = chunk_documents(documents, chunk_size=1024, chunk_overlap=200)

print(f"\nChunk Size Comparison:")
print(f"Small chunks (256): {len(chunks_small)} chunks")
print(f"Medium chunks (512): {len(chunks_medium)} chunks")
print(f"Large chunks (1024): {len(chunks_large)} chunks")

Total documents: 12

Chunk Size Comparison:
Small chunks (256): 60 chunks
Medium chunks (512): 36 chunks
Large chunks (1024): 12 chunks


In [8]:
# Embedding Model 1: all-MiniLM-L6-v2 (384 dimensions)
embedding_model_1 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# Embedding Model 2: all-mpnet-base-v2 (768 dimensions)
embedding_model_2 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

print("Embedding Models Loaded:")
print("1. all-MiniLM-L6-v2 (384 dimensions) - Faster, smaller")
print("2. all-mpnet-base-v2 (768 dimensions) - More accurate, larger")

# Test embeddings
test_text = "What is anxiety?"
emb1 = embedding_model_1.embed_query(test_text)
emb2 = embedding_model_2.embed_query(test_text)
print(f"\nEmbedding dimensions: Model 1 = {len(emb1)}, Model 2 = {len(emb2)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Models Loaded:
1. all-MiniLM-L6-v2 (384 dimensions) - Faster, smaller
2. all-mpnet-base-v2 (768 dimensions) - More accurate, larger

Embedding dimensions: Model 1 = 384, Model 2 = 768


In [9]:
# Vector Database 1: ChromaDB with embedding model 1
print("Creating ChromaDB vector store...")
chroma_db = Chroma.from_documents(
    documents=chunks_medium,
    embedding=embedding_model_1,
    collection_name="mental_health_chroma"
)

# Vector Database 2: FAISS with embedding model 2
print("Creating FAISS vector store...")
faiss_db = FAISS.from_documents(
    documents=chunks_medium,
    embedding=embedding_model_2
)

print("\n✅ Vector Databases Created:")
print("1. ChromaDB with MiniLM embeddings (384d)")
print("2. FAISS with MPNet embeddings (768d)")

Creating ChromaDB vector store...
Creating FAISS vector store...

✅ Vector Databases Created:
1. ChromaDB with MiniLM embeddings (384d)
2. FAISS with MPNet embeddings (768d)


In [10]:
# Using TinyLlama - a small 1.1B parameter model
# This clearly shows the difference RAG makes with limited model knowledge

from huggingface_hub import login

# Login to HuggingFace (optional, for gated models)
# login(token="your_token_here")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# Create text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15
)

print(f"✅ Model loaded: TinyLlama 1.1B parameters")
print(f"Device: {next(model.parameters()).device}")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'temperature', 'top_p', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Model loaded: TinyLlama 1.1B parameters
Device: cpu


In [11]:
def generate_without_rag(query):
    """Generate response WITHOUT RAG (baseline)"""
    prompt = f"""<|system|>
You are a mental health assistant. Answer the question directly based on your knowledge.
</s>
<|user|>
{query}
</s>
<|assistant|>
"""
    response = llm_pipeline(prompt)[0]['generated_text']
    # Extract only the assistant's response
    assistant_response = response.split("<|assistant|>")[-1].strip()
    return assistant_response


def generate_with_rag(query, vector_db, db_name="Database", top_k=3):
    """Generate response WITH RAG"""
    # Retrieve relevant documents
    retrieved_docs = vector_db.similarity_search(query, k=top_k)

    # Format context from retrieved documents
    context = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])

    # Create RAG prompt
    prompt = f"""<|system|>
You are a mental health assistant. Answer the question using ONLY the information provided in the context below.
If the context doesn't contain relevant information, say so.
</s>
<|user|>
Context from {db_name}:
{context}

Question: {query}
</s>
<|assistant|>
"""
    response = llm_pipeline(prompt)[0]['generated_text']
    assistant_response = response.split("<|assistant|>")[-1].strip()

    return assistant_response, retrieved_docs


def display_results(query, without_rag, with_rag_response, retrieved_docs, db_name):
    """Display formatted comparison results"""
    print("=" * 80)
    print(f"📝 QUERY: {query}")
    print("=" * 80)

    print("\n" + "─" * 40)
    print("❌ WITHOUT RAG (Baseline LLM Only)")
    print("─" * 40)
    print(without_rag)

    print("\n" + "─" * 40)
    print(f"✅ WITH RAG (Using {db_name})")
    print("─" * 40)
    print(with_rag_response)

    print("\n" + "─" * 40)
    print(f"📚 RETRIEVED DOCUMENTS FROM {db_name}:")
    print("─" * 40)
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\n[Document {i}] - Topic: {doc.metadata.get('topic', 'N/A')}")
        print(f"{doc.page_content[:300]}...")
    print("\n" + "=" * 80)

In [12]:
query1 = "What are effective techniques for managing panic attacks?"

print("\n🔬 EXPERIMENT 1: Panic Attack Management\n")

# Without RAG
response_no_rag = generate_without_rag(query1)

# With RAG - ChromaDB
response_chroma, docs_chroma = generate_with_rag(query1, chroma_db, "ChromaDB")

# Display results
display_results(query1, response_no_rag, response_chroma, docs_chroma, "ChromaDB")


🔬 EXPERIMENT 1: Panic Attack Management

📝 QUERY: What are effective techniques for managing panic attacks?

────────────────────────────────────────
❌ WITHOUT RAG (Baseline LLM Only)
────────────────────────────────────────
Here are some effective techniques for managing panic attacks:

1. Deep breathing exercises: Taking slow, deep breaths and focusing on your diaphragm can help calm your body and mind during an attack.

2. Progressive muscle relaxation: Repeating specific muscles you want to relax one at a time can help release tension and reduce feelings of anxiety or fear.

3. Mindfulness meditation: Focusing on your thoughts and sensations without judgment can help bring awareness to your current state of mind and provide a sense of control.

4. Visualization: Imagine a safe and peaceful environment that calms your nerves. This technique can be especially helpful if you're having difficulty visualizing a place that makes you feel more comfortable.

5. Exercise: Physical activity

In [13]:
query2 = "What are the main treatment options for depression?"

print("\n🔬 EXPERIMENT 2: Depression Treatment\n")

# Without RAG
response_no_rag = generate_without_rag(query2)

# With RAG - FAISS
response_faiss, docs_faiss = generate_with_rag(query2, faiss_db, "FAISS")

# Display results
display_results(query2, response_no_rag, response_faiss, docs_faiss, "FAISS")


🔬 EXPERIMENT 2: Depression Treatment

📝 QUERY: What are the main treatment options for depression?

────────────────────────────────────────
❌ WITHOUT RAG (Baseline LLM Only)
────────────────────────────────────────
There are several main treatment options available for depression, which may vary depending on individual patient needs and preferences:

1. Antidepressant medications (e.g., selective serotonin reuptake inhibitors [SSRIs], monoamine oxidase inhibitors [MAOI]-based antidepressants)
2. Psychotherapy (e.g., cognitive behavioral therapy [CBT], exposure therapy)
3. Cognitive-behavioral therapy (CBT): This involves teaching patients how to identify negative thought patterns and replace them with positive ones. It can be effective for treating depression by improving mood and reducing symptoms.
4. Light therapy (also known as phototherapy or tanning bed therapy): This involves using artificial light for a set period of time, usually daily or weekly, to improve mood and reduce sy

In [14]:
query3 = "Explain Cognitive Behavioral Therapy and its effectiveness"

print("\n🔬 EXPERIMENT 3: CBT Therapy\n")

# Without RAG
response_no_rag = generate_without_rag(query3)

# With RAG - ChromaDB
response_rag, docs_rag = generate_with_rag(query3, chroma_db, "ChromaDB")

# Display results
display_results(query3, response_no_rag, response_rag, docs_rag, "ChromaDB")


🔬 EXPERIMENT 3: CBT Therapy

📝 QUERY: Explain Cognitive Behavioral Therapy and its effectiveness

────────────────────────────────────────
❌ WITHOUT RAG (Baseline LLM Only)
────────────────────────────────────────
Cognitive Behavioral Therapy (CBT) is an evidence-based treatment for depression, anxiety disorders, and other mood disorders. It involves identifying negative thoughts or beliefs that contribute to symptoms, exposing those thoughts to challenge and replacing them with positive, realistic ones. The process of CBT involves several steps:

1. Assessment - A therapist will ask you questions about your thoughts and behaviors related to your mood disorder. They may also conduct a physical examination if necessary.

2. Identification - Based on the assessment, the therapist will identify any cognitive distortions or maladaptive patterns in thought, behavior, or perception that have contributed to your current mood disorder.

3. Challenge - Through individual or group sessions, the

In [ ]:
query4 = "How does sleep affect mental health and what is CBT-I?"

print("\n🔬 EXPERIMENT 4: Sleep and Mental Health\n")

# Without RAG
response_no_rag = generate_without_rag(query4)

# With RAG - FAISS
response_rag, docs_rag = generate_with_rag(query4, faiss_db, "FAISS")

# Display results
display_results(query4, response_no_rag, response_rag, docs_rag, "FAISS")

In [ ]:
query5 = "What is PTSD and how is EMDR used to treat it?"

print("\n🔬 EXPERIMENT 5: Vector Database Comparison (ChromaDB vs FAISS)\n")
print("=" * 80)
print(f"📝 QUERY: {query5}")
print("=" * 80)

# Without RAG
response_no_rag = generate_without_rag(query5)
print("\n❌ WITHOUT RAG:")
print(response_no_rag)

# With ChromaDB
response_chroma, docs_chroma = generate_with_rag(query5, chroma_db, "ChromaDB", top_k=2)
print("\n" + "─" * 40)
print("✅ WITH RAG - ChromaDB (MiniLM-L6 Embeddings):")
print(response_chroma)

# With FAISS
response_faiss, docs_faiss = generate_with_rag(query5, faiss_db, "FAISS", top_k=2)
print("\n" + "─" * 40)
print("✅ WITH RAG - FAISS (MPNet Embeddings):")
print(response_faiss)

print("\n" + "─" * 40)
print("📊 RETRIEVAL COMPARISON:")
print("─" * 40)
print("\nChromaDB Retrieved Topics:", [d.metadata.get('topic') for d in docs_chroma])
print("FAISS Retrieved Topics:", [d.metadata.get('topic') for d in docs_faiss])

In [15]:
from collections import Counter

def evaluate_response_quality(response, context_docs):
    """Simple evaluation metrics"""
    # Check if response uses information from retrieved docs
    response_lower = response.lower()

    # Count keyword matches from retrieved documents
    keywords_found = 0
    total_keywords = 0

    for doc in context_docs:
        # Extract key terms from documents
        doc_words = set(doc.page_content.lower().split())
        important_words = [w for w in doc_words if len(w) > 5]
        total_keywords += len(important_words)
        keywords_found += sum(1 for w in important_words if w in response_lower)

    relevance_score = keywords_found / max(total_keywords, 1) * 100

    return {
        "response_length": len(response.split()),
        "relevance_score": round(relevance_score, 2),
        "keywords_matched": keywords_found
    }

# Evaluation across test queries
test_queries = [
    "What are the symptoms of anxiety?",
    "How can mindfulness help with stress?",
    "What should I do during a mental health crisis?"
]

print("\n📊 QUANTITATIVE EVALUATION\n")
print("=" * 80)

results = []
for query in test_queries:
    # Without RAG
    no_rag = generate_without_rag(query)

    # With RAG
    with_rag, docs = generate_with_rag(query, chroma_db, "ChromaDB")

    # Evaluate
    no_rag_metrics = {"response_length": len(no_rag.split()), "relevance_score": "N/A", "keywords_matched": "N/A"}
    rag_metrics = evaluate_response_quality(with_rag, docs)

    results.append({
        "Query": query[:40] + "...",
        "No-RAG Words": no_rag_metrics["response_length"],
        "RAG Words": rag_metrics["response_length"],
        "RAG Relevance %": rag_metrics["relevance_score"],
        "Keywords Matched": rag_metrics["keywords_matched"]
    })

results_df = pd.DataFrame(results)
display(results_df)


📊 QUANTITATIVE EVALUATION



,Query,No-RAG Words,RAG Words,RAG Relevance %,Keywords Matched
0,What are the symptoms of anxiety?...,216,63,23.53,16
1,How can mindfulness help with stress?...,139,178,35.79,34
2,What should I do during a mental health ...,227,95,37.84,14


In [16]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                        RAG SYSTEM EVALUATION SUMMARY                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  SYSTEM CONFIGURATION                                                        ║
║  ─────────────────────                                                       ║
║  • LLM: TinyLlama 1.1B (small model to highlight RAG benefits)              ║
║  • Embedding Model 1: all-MiniLM-L6-v2 (384 dimensions)                     ║
║  • Embedding Model 2: all-mpnet-base-v2 (768 dimensions)                    ║
║  • Vector DB 1: ChromaDB                                                     ║
║  • Vector DB 2: FAISS                                                        ║
║  • Chunk Size: 512 tokens with 100 token overlap                            ║
║  • Domain: Mental Health Knowledge Base (12 documents)                       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  KEY OBSERVATIONS                                                            ║
║  ─────────────────                                                           ║
║  1. WITHOUT RAG: Small LLM produces generic, sometimes inaccurate answers   ║
║  2. WITH RAG: Responses are grounded in specific, accurate domain knowledge ║
║  3. RAG enables small models to provide expert-level domain responses       ║
║  4. Both ChromaDB and FAISS effectively retrieve relevant documents         ║
║  5. MPNet embeddings (768d) may provide slightly better semantic matching   ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  COMPONENTS DEMONSTRATED                                                     ║
║  ──────────────────────                                                      ║
║  ✓ Document ingestion and preprocessing                                      ║
║  ✓ Text chunking with multiple strategies (256, 512, 1024)                  ║
║  ✓ Multiple embedding models (MiniLM, MPNet)                                ║
║  ✓ Two vector databases (ChromaDB, FAISS)                                   ║
║  ✓ Semantic search and retrieval                                            ║
║  ✓ Prompt engineering with retrieved context                                ║
║  ✓ Small parameter LLM (TinyLlama 1.1B)                                     ║
║  ✓ Comparative evaluation with/without RAG                                  ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════════════╗
║                        RAG SYSTEM EVALUATION SUMMARY                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  SYSTEM CONFIGURATION                                                        ║
║  ─────────────────────                                                       ║
║  • LLM: TinyLlama 1.1B (small model to highlight RAG benefits)              ║
║  • Embedding Model 1: all-MiniLM-L6-v2 (384 dimensions)                     ║
║  • Embedding Model 2: all-mpnet-base-v2 (768 dimensions)                    ║
║  • Vector DB 1: ChromaDB                                                     ║
║  • Vector DB 2: FAISS                                                        ║
║  • Chunk Size: 512 tokens with 100 token overlap                            ║
║  • Domain: Mental Health Knowledge Base (12 documents)                       ║
╠═══════════════════════════════